# Eval-Driven Development for Agents — Build-Along
## (Offline Loop): From a Caught Trace to a Passing CI

In the next ~40 minutes we'll build an offline reliability loop end-to-end. Every cell here runs on your machine. By the end you'll have:

1. **A dataset** in LangSmith (4 strategic examples)
2. **An agent** with HITL middleware
3. **Evaluators** — Code + LLM-as-judge
4. **Experiments** — running the dataset through the agent, scored
5. **A regression caught live** — edit prompt → see the score drop
6. **A CI gate** — pytest in GitHub Actions
7. **A calibration loop** — annotation queue for human review

> **Today's framing.** This is Module 4 of a 6-module agent-engineering arc:
>
> 1. `create_agent` fundamentals · 2. Tracing & observability · 3. Prompt engineering + Playground
>
> **▶ 4. Eval-driven development (offline loop) — TODAY ◀** · 5. Online evals + Insights Agent · 6. Multi-agent

> *Reference:* [LangChain Agent Evaluation Readiness Checklist](https://www.langchain.com/blog/agent-evaluation-readiness-checklist) · [LangSmith Evaluation Quickstart](https://docs.langchain.com/langsmith/evaluation-quickstart) · [openevals](https://docs.langchain.com/langsmith/openevals)

Q&A is welcome throughout. Let's build.

In [34]:
# Setup — load env, instantiate LangSmith client, import LangChain primitives.
# Required env vars (in .env): OPENAI_API_KEY, LANGSMITH_API_KEY,
# LANGSMITH_TRACING=true, LANGSMITH_PROJECT=agent-eval-buildalong
import os
import datetime
from dotenv import load_dotenv
load_dotenv()

from langsmith import Client, evaluate
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

# Examples are hand-curated in src/seed_data.py — 21 regression + 3 capability.
# Import them; we'll show the structure and the key examples we'll focus on
# in the next cell.
from src.seed_data import SEED_EXAMPLES, CAPABILITY_EXAMPLES

client = Client()
print(f"LangSmith project: {os.getenv('LANGSMITH_PROJECT', '(unset)')}")
print(f"Loaded {len(SEED_EXAMPLES)} regression examples + {len(CAPABILITY_EXAMPLES)} capability examples from src/seed_data.py")

LangSmith project: agent-eval-test
Loaded 21 regression examples + 3 capability examples from src/seed_data.py


## Step 1 of 4 — Build the datasets

A **dataset** in LangSmith is a persistent named collection of **examples**. Two complementary suites per the [readiness checklist](https://www.langchain.com/blog/agent-evaluation-readiness-checklist):

- **Regression** (`support-triage-buildalong`) — **21 examples** of known-correct behavior. Pass rate ~100%; gates CI.
- **Capability** (`support-triage-buildalong-capability`) — **3 deliberately hard examples**. Pass rate &lt;100% by design; tracks improvement over time without gating.

### Example structure (every example follows this shape)

```python
{
    "id": "ex-001",                           # stable join key — preserved across upserts
    "inputs": {"message": "..."},             # what the agent receives
    "reference_outputs": {                    # what we expect from the agent
        "expected_classification": "account",
        "expected_tools": ["get_customer_plan", "search_kb"],
        "expected_kb_doc_id": "kb-002",
        "should_escalate": False,
        "should_refund": False,
    },
    "split": ["regression"],                  # optional — tags subsets for filtering / targeted eval runs
}
```

Hand-curated in `src/seed_data.py` (in production you'd start with ~20-50 real agent traces; the readiness checklist's first step). Same shape for both regression and capability examples.

### Splits — tagging subsets

Two splits are tagged on the regression dataset:

- **`regression`** (4 examples: `ex-001`, `ex-005`, `ex-008`, `ex-012`) — the fast smoke set the CI gate runs on every PR. Single source of truth for "what gates a merge."
- **`trap`** (4 examples: `ex-005`, `ex-008`, `ex-012`, `ex-019`) — the canonical pedagogical traps where evaluation earns its keep. Three of them overlap with the regression split.
- **`base`** (the remaining 16 examples) — broader coverage; not gating CI but available for ad-hoc deep dives.

The full dataset is upserted; the demo and CI both target the `regression` split.

### Examples we'll focus on in today's demo

| ID | Split | Type | Message | Tests |
|---|---|---|---|---|
| `ex-001` | `regression` | Happy path | *"I forgot my password"* (free-tier) | Cite `kb-002`; no refund, no escalation |
| `ex-005` | `regression`, `trap` | Tricky billing | *"When does my next billing cycle start?"* | Classify as billing, not refund |
| `ex-008` | `regression`, `trap` | Angry customer | *"This is RIDICULOUS. I'm furious."* | Must call `escalate` |
| `ex-012` | `regression`, `trap` | **THE TRAP** — *"I want a refund"* (free-tier) | Must NOT call `issue_refund` — drives the regression demo |
| `cap-001` | (capability dataset) | Capability | *"3 issues in one message — refund + password + 401 errors"* | Multi-issue handling — agent should address all three or escalate |

The regression dataset gates CI; the capability dataset tracks improvement. Both run in the same pytest invocation in CI — the split is at the assertion layer, not the runner.

In [35]:
REGRESSION_DATASET = "support-triage-buildalong"
CAPABILITY_DATASET = "support-triage-buildalong-capability"


def upsert_dataset(name: str, description: str, examples: list) -> None:
    """Idempotent upsert keyed on ex_id metadata.
    
    Re-runs preserve example UUIDs (required for LangSmith Compare view to
    join experiments across re-runs). Threads the optional `split` field
    through so subsets are filterable in the UI and targetable via
    list_examples(splits=[...]).
    """
    try:
        ds = client.read_dataset(dataset_name=name)
        print(f"  '{name}' exists; updating examples in place.")
    except Exception:
        ds = client.create_dataset(dataset_name=name, description=description)
        print(f"  Created '{name}'")
    
    existing = {e.metadata.get("ex_id"): e for e in client.list_examples(dataset_id=ds.id)
                if e.metadata and e.metadata.get("ex_id")}
    
    to_create = []
    for ex in examples:
        # `split` is a LangSmith primitive for tagging subsets — filterable in
        # the dataset UI, targetable via list_examples(splits=[...]).
        split = ex.get("split")
        if ex["id"] in existing:
            client.update_example(
                example_id=existing[ex["id"]].id,
                inputs=ex["inputs"], outputs=ex["reference_outputs"],
                metadata={"ex_id": ex["id"]},
                split=split,
            )
        else:
            payload = {
                "inputs": ex["inputs"], "outputs": ex["reference_outputs"],
                "metadata": {"ex_id": ex["id"]},
            }
            if split is not None:
                payload["split"] = split
            to_create.append(payload)
    if to_create:
        client.create_examples(dataset_id=ds.id, examples=to_create)
    print(f"  → {len(examples)} examples upserted.\n")


print("Upserting both datasets…\n")
upsert_dataset(REGRESSION_DATASET, "Build-along regression dataset (21 examples)", SEED_EXAMPLES)
upsert_dataset(CAPABILITY_DATASET, "Build-along capability dataset (3 hard examples)", CAPABILITY_EXAMPLES)

print(f"✓ Done. Open in LangSmith → Datasets to verify both appear.")
print(f"  In '{REGRESSION_DATASET}' → Examples tab, the Splits filter shows:")
print(f"    • regression (4 examples) — the fast smoke set CI gates on")
print(f"    • trap (4 examples) — the canonical pedagogical traps (ex-005/008/012/019)")
print(f"    • base (default) — the rest of the dataset")

Upserting both datasets…

  'support-triage-buildalong' exists; updating examples in place.
  → 21 examples upserted.

  'support-triage-buildalong-capability' exists; updating examples in place.
  → 3 examples upserted.

✓ Done. Open in LangSmith → Datasets to verify both appear.
  In 'support-triage-buildalong' → Examples tab, the Splits filter shows:
    • regression (4 examples) — the fast smoke set CI gates on
    • trap (4 examples) — the canonical pedagogical traps (ex-005/008/012/019)
    • base (default) — the rest of the dataset


## Step 2 of 4 — Build the agent

The agent is one `create_agent(...)` call. Three pieces:

- **Model** — `openai:gpt-4o-mini`
- **Tools** — 4 functions: `get_customer_plan`, `search_kb`, `issue_refund`, `escalate`
- **System prompt** — encodes the policy (including the safety rule we'll later remove to demonstrate the regression catch)
- **Middleware** — `HumanInTheLoopMiddleware` interrupts on `issue_refund` (the irreversible action)

Notice the policy is in the prompt, not the code. That's the **change surface** we'll exercise.

In [36]:
# --- Tools (4) ---
@tool
def get_customer_plan(customer_id: str) -> dict:
    """Return a customer's plan and account details."""
    plans = {
        "cust_001": {"plan": "pro", "monthly_fee": 49, "signup_date": "2024-01-15"},
        "cust_002": {"plan": "free", "monthly_fee": 0, "signup_date": "2024-09-01"},
    }
    return plans.get(customer_id, {"error": "customer not found"})

@tool
def search_kb(query: str) -> dict:
    """Search the knowledge base for the relevant policy/answer."""
    kb = {
        "kb-001": "Refund policy: paid-plan customers eligible if account < 30 days; never for free tier.",
        "kb-002": "Password reset: visit /account/reset; link expires in 1 hour.",
        "kb-003": "Escalation: angry/repeat-contact customers go to a human agent.",
    }
    if "refund" in query.lower(): return {"doc_id": "kb-001", "content": kb["kb-001"]}
    if "password" in query.lower(): return {"doc_id": "kb-002", "content": kb["kb-002"]}
    if "escalat" in query.lower(): return {"doc_id": "kb-003", "content": kb["kb-003"]}
    return {"doc_id": None, "content": "No matching KB doc found."}

@tool
def issue_refund(customer_id: str, amount: float, reason: str) -> str:
    """Submit a refund REQUEST (not execute). Customer is told the request is submitted."""
    return f"Refund request submitted for {customer_id}: ${amount} ({reason}). Customer will receive an email."

@tool
def escalate(customer_id: str, reason: str) -> str:
    """Escalate to a human agent."""
    return f"Escalated {customer_id}: {reason}"

# --- System prompt (the policy) ---
SYSTEM_PROMPT = """You are a customer support triage agent for Acme SaaS.

For every customer message, you MUST:
1. Classify the issue as one of: billing, technical, account, other
2. Use get_customer_plan to fetch the customer's plan and details
3. Use search_kb to look up the relevant policy/answer in the knowledge base
4. Resolve via one of:
   - reply with the KB-grounded answer
   - escalate if the issue is angry, complex, or out of scope
   - issue_refund if (and only if) the customer is on a paid plan AND has a legitimate refund-eligible issue per kb-001

You MUST include the KB document ID inside the \"answer\" field (e.g., \"...per kb-001\") whenever you used search_kb.
When you used issue_refund, your \"answer\" must tell the customer their refund request has been submitted for review and they will receive an email once it is processed — do NOT claim the refund has already been issued.
Be concise. Never refund a free-tier customer.

Output your final response in this exact JSON format on the last line:
{\"classification\": \"<billing|technical|account|other>\", \"answer\": \"<your answer>\"}
"""

def build_agent(system_prompt: str = SYSTEM_PROMPT):
    """Build a fresh agent with a given system prompt. Letting us swap prompts later."""
    return create_agent(
        model="openai:gpt-4o-mini",
        tools=[get_customer_plan, search_kb, issue_refund, escalate],
        system_prompt=system_prompt,
        middleware=[HumanInTheLoopMiddleware(interrupt_on={"issue_refund": True})],
        checkpointer=InMemorySaver(),
    )

agent = build_agent()
print("✓ Agent built — gpt-4o-mini + 4 tools + HITL on issue_refund.")

✓ Agent built — gpt-4o-mini + 4 tools + HITL on issue_refund.


In [37]:
# Quick sanity check: run the agent on the trap example to verify HITL fires.
trap = next(e for e in SEED_EXAMPLES if e["id"] == "ex-012")
config = {"configurable": {"thread_id": "sanity-trap"}}

result = agent.invoke(
    {"messages": [{"role": "user", "content": trap["inputs"]["message"]}]},
    config=config, version="v2",
)
print("Interrupted?", bool(result.interrupts), "  ← HITL fired if True")
print("Last message:", result.value["messages"][-1].content[:200])

Interrupted? False   ← HITL fired if True
Last message: {"classification": "billing", "answer": "As you are on a free plan, I am unable to process a refund. Refunds are only available for paid plans."}


## Step 3 of 4 — Build the evaluators

An **evaluator** is a function that scores an agent run against a reference. LangSmith documents 5 canonical types — **Code, LLM-as-judge, Pairwise, Composite, Summary**. We'll write 2 of these.

All evaluators follow the same canonical signature from the [Evaluation Quickstart](https://docs.langchain.com/langsmith/evaluation-quickstart):

```python
def evaluator(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    return {"key": "...", "score": 0|1|0.5|None, "comment": "..."}
```

Same shape — different machinery underneath. The runner doesn't know which type it's executing.

In [38]:
# --- Code evaluators (4 of them) ---
# All follow the canonical (inputs, outputs, reference_outputs) -> {key, score, comment} signature.
# Deterministic Python ~10 lines each. Fast, free, the right shape for structural properties.

import re


def refund_safety(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    """THE LOAD-BEARING CI GATE — the only evaluator that hard-asserts in pytest.
    
    Why this one gates: refunds are irreversible (financial liability). The split
    between gating and tracking is risk-weighted, not arbitrary.
    """
    tools_called = outputs.get("tool_calls", [])
    refunded = "issue_refund" in tools_called
    should_refund = reference_outputs.get("should_refund", False)
    if refunded and not should_refund:
        return {"key": "refund_safety", "score": 0,
                "comment": "Agent issued refund when it should not have"}
    if not refunded and should_refund:
        return {"key": "refund_safety", "score": 0,
                "comment": "Agent failed to issue refund when it should have"}
    return {"key": "refund_safety", "score": 1}


def classification_correct(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    """Extracts the JSON classification from the agent's final message and compares to expected.
    Catches: agent decides the wrong category (e.g. classifies billing as technical).
    """
    expected = reference_outputs.get("expected_classification")
    if not expected:
        return {"key": "classification_correct", "score": None}
    final_msg = outputs.get("final_message", "")
    m = re.search(r'"classification":\s*"(\w+)"', final_msg)
    if not m:
        return {"key": "classification_correct", "score": 0,
                "comment": "no classification in output"}
    actual = m.group(1)
    return {"key": "classification_correct", "score": 1 if actual == expected else 0,
            "comment": f"expected={expected} actual={actual}"}


def escalation_correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    """Checks whether escalate was called when it should have been (and vice versa).
    Mirrors refund_safety's structure but for the escalate tool — same shape, different concern.
    """
    tools_called = outputs.get("tool_calls", [])
    escalated = "escalate" in tools_called
    should_escalate = reference_outputs.get("should_escalate", False)
    if escalated and not should_escalate:
        return {"key": "escalation_correctness", "score": 0,
                "comment": "Agent escalated when it should not have"}
    if not escalated and should_escalate:
        return {"key": "escalation_correctness", "score": 0,
                "comment": "Agent failed to escalate when it should have"}
    return {"key": "escalation_correctness", "score": 1}


def trajectory_superset(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    """Code evaluator on the TRAJECTORY DIMENSION.
    
    'Trajectory' is a dimension of what's evaluated (per the readiness checklist:
    final response, trajectory, state changes), NOT an evaluator type.
    
    This evaluator's job is to score the path: does the agent's tool sequence
    include every required tool? Catches cases where the agent reaches a
    correct-looking answer through a wrong path.
    """
    expected_tools = reference_outputs.get("expected_tools", [])
    if not expected_tools:
        return {"key": "trajectory_superset", "score": None, "comment": "no reference"}
    actual = outputs.get("tool_calls", [])
    score = 1 if all(t in actual for t in expected_tools) else 0
    return {"key": "trajectory_superset", "score": score,
            "comment": f"expected={expected_tools} actual={actual}"}


print("✓ 4 code evaluators defined: refund_safety (CI gate), classification_correct,")
print("  escalation_correctness, trajectory_superset (trajectory dimension)")

✓ 4 code evaluators defined: refund_safety (CI gate), classification_correct,
  escalation_correctness, trajectory_superset (trajectory dimension)


In [39]:
# LLM-as-judge — kb_grounding_judge.
# Wraps a rubric prompt + reasoning model (o3-mini) via openevals.create_llm_as_judge.
# Why o3-mini specifically: reasoning models are more stable on subjective rubrics
# than gpt-4o for the cost; trade-off is latency (~5s per call).
from openevals.llm import create_llm_as_judge
from openai import LengthFinishReasonError

# Note on the prompt format: openevals.create_llm_as_judge formats the prompt
# via Python's str.format(). Use TOP-LEVEL placeholders {inputs}, {outputs},
# {reference_outputs} — the library passes the full dicts; subscript syntax
# like {inputs[message]} causes "TypeError: string indices must be integers".
KB_GROUNDING_PROMPT = """You are evaluating whether an AI customer support agent's answer is grounded in the knowledge base.

The agent must cite a KB doc ID (e.g., "kb-001") when it uses information from the knowledge base.

<inputs>
{inputs}
</inputs>

<agent_answer>
{outputs}
</agent_answer>

<expected_kb_doc>
{reference_outputs}
</expected_kb_doc>

Score the answer 0.0 to 1.0:
- 1.0: Cited the expected KB doc and the answer is consistent with it
- 0.5: Cited a KB doc but it was the wrong one, OR cited the right one but the answer is loose
- 0.0: Did not cite any KB doc, or made up information

Respond ONLY with the score as a float. No explanation.
"""


def kb_grounding_judge(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    """LLM-as-judge for KB grounding via openevals + o3-mini."""
    # 1. Early skip — examples without expected citation return None, not 0.
    #    Otherwise the headline average tanks.
    if not reference_outputs.get("expected_kb_doc_id"):
        return {"key": "kb_grounding", "score": None,
                "comment": "skipped: no KB citation expected"}
    # 2. The judge — openevals wraps prompt + model with proper LangSmith tracing.
    evaluator = create_llm_as_judge(
        prompt=KB_GROUNDING_PROMPT,
        model="openai:o3-mini",
        feedback_key="kb_grounding",
    )
    # 3. LengthFinishReasonError guard — caught a real o3-mini runaway during
    #    rehearsal (99,998 completion tokens). Without this, one bad example
    #    hangs the eval for ~10 minutes.
    try:
        return evaluator(inputs=inputs, outputs=outputs, reference_outputs=reference_outputs)
    except LengthFinishReasonError as e:
        return {"key": "kb_grounding", "score": None,
                "comment": f"skipped: o3-mini token limit ({e})"}


print("✓ LLM-as-judge defined: kb_grounding_judge (o3-mini)")
print("\nWhy LLM-as-judge at all: code can check structural properties")
print("(was the right kb-id cited?) but can't read whether the response")
print("is faithful to the citation. Subjective dimensions need a model.")

✓ LLM-as-judge defined: kb_grounding_judge (o3-mini)

Why LLM-as-judge at all: code can check structural properties
(was the right kb-id cited?) but can't read whether the response
is faithful to the citation. Subjective dimensions need a model.


## Step 4 of 4 — Run experiments

An **experiment** is `evaluate(target, data, evaluators=[...])`. Each call:
1. Iterates the dataset, invoking the target on each example
2. Runs each evaluator on the result
3. Logs scores as **feedback records** attached to runs
4. All grouped under a named experiment in the dataset's Experiments tab

The `target` is a function that takes example inputs and returns agent outputs in the shape evaluators expect.

### Targeting a split

We upserted **all 21 examples** so they're all visible in LangSmith. But the regression demo runs only the **4 examples in the `regression` split** — same set the pytest CI gate runs on. That's the single-source-of-truth alignment: the dataset is the spec, and the `regression` split is the contract for "what gates a PR."

```python
data = client.list_examples(dataset_name=..., splits=["regression"])
evaluate(target, data=data, ...)
```

This makes the v1/v2/v3 demo run in ~30 sec instead of ~90 sec, and it's the same loop you'd run locally before opening a PR.

In [40]:
# The target function — bridges 'dataset example' to 'agent output'.
# Returns the shape evaluators expect: {final_message, tool_calls}.
#
# Handles BOTH input shapes in the dataset:
#   - single-turn:  {"message": "..."}              ← 20 of the 21 examples
#   - multi-turn:   {"messages": ["...", "..."]}    ← ex-021 only (Q&A insurance)
# Multi-turn replays each user message through the same thread_id so the
# agent's checkpointer carries state across turns. The same evaluators score
# the final result either way.
def make_target(agent_instance):
    def target(inputs: dict) -> dict:
        # Detect input shape
        if "messages" in inputs and isinstance(inputs["messages"], list):
            user_turns = inputs["messages"]
        else:
            user_turns = [inputs["message"]]

        config = {"configurable": {"thread_id": f"eval-{abs(hash(user_turns[0]))}"}}
        result = None
        for user_msg in user_turns:
            result = agent_instance.invoke(
                {"messages": [{"role": "user", "content": user_msg}]},
                config=config, version="v2",
            )
            # Auto-approve any HITL interrupts — we measure agent INTENT,
            # not human review. In production a real reviewer would approve/reject.
            while result.interrupts:
                result = agent_instance.invoke(
                    Command(resume={"decisions": [{"type": "approve"}]}),
                    config=config, version="v2",
                )

        messages = result.value["messages"]
        return {
            "final_message": messages[-1].content,
            "tool_calls": [tc["name"] for m in messages
                           for tc in (getattr(m, "tool_calls", None) or [])],
        }
    return target


# All 5 evaluators — runner-agnostic, same canonical signature.
ALL_EVALUATORS = [
    refund_safety,
    classification_correct,
    escalation_correctness,
    trajectory_superset,
    kb_grounding_judge,
]

# Run experiment v1: agent with the safety guardrail intact.
# Filtered to the `regression` split — 4 examples (the CI fast-set), not all 21.
# The full dataset is upserted (and visible in LangSmith); the demo runs the
# subset CI gates on. ~30 sec wall clock with concurrency=4.
RUN_ID = datetime.datetime.now().strftime("%Y%m%d-%H%M")

regression_data = list(client.list_examples(
    dataset_name=REGRESSION_DATASET, splits=["regression"]
))
print(f"Regression split has {len(regression_data)} examples: "
      f"{[e.metadata.get('ex_id') for e in regression_data]}\n")

results_v1 = evaluate(
    make_target(agent),
    data=regression_data,
    evaluators=ALL_EVALUATORS,
    experiment_prefix=f"v1-baseline-{RUN_ID}",
    metadata={"run_id": RUN_ID, "variant": "baseline", "demo_set": "buildalong"},
    max_concurrency=4,
)
print(f"\n✓ Experiment v1-baseline-{RUN_ID} done.")
print(f"  Open: Datasets → {REGRESSION_DATASET} → Experiments tab")
print(f"  All 5 evaluators ran on the regression split (4 examples) — should see 100% pass rate (this is the baseline).\n")

# Inline experiment view — same pattern as the intro-to-langsmith experiments
# notebook. ExperimentResults renders as an HTML table in Jupyter; .to_pandas()
# gives the richer per-example score frame (one row per example, one column
# per evaluator + one for inputs/outputs).
results_v1.to_pandas()

Regression split has 4 examples: ['ex-012', 'ex-008', 'ex-005', 'ex-001']

View the evaluation results for experiment: 'v1-baseline-20260506-2335-c717256f' at:
https://smith.langchain.com/o/bcd92ec8-b810-4fa0-a0c0-a6455008401d/datasets/cffad3f2-d09b-43aa-b6df-3f8f7c7dcecf/compare?selectedSessions=3be59d1a-e426-4893-87b6-bfbe9b869a3b




0it [00:00, ?it/s]


✓ Experiment v1-baseline-20260506-2335 done.
  Open: Datasets → support-triage-buildalong → Experiments tab
  All 5 evaluators ran on the regression split (4 examples) — should see 100% pass rate (this is the baseline).



,inputs.message,outputs.final_message,outputs.tool_calls,error,reference.should_refund,reference.should_escalate,reference.expected_classification,feedback.refund_safety,feedback.classification_correct,feedback.escalation_correctness,feedback.trajectory_superset,feedback.kb_grounding,execution_time,example_id,id,reference.expected_kb_doc_id,reference.expected_tools
0,This is RIDICULOUS. I'm cust_003 and your serv...,"{""classification"": ""other"", ""answer"": ""Your co...","[get_customer_plan, escalate, escalate]",None,False,True,technical,1,0,1,NaN,None,2.954293,65736ce5-ec14-4b3d-a6b1-fc1a33a460ba,019dff6f-061a-79d1-9a3e-580c58df1055,NaN,NaN
1,I'm cust_002 and I want a refund for last month.,"{""classification"": ""billing"", ""answer"": ""I'm s...",[get_customer_plan],None,False,NaN,billing,1,1,1,NaN,False,1.846366,0cd4d8c0-40f2-4e83-9b0b-00c7ecbbd022,019dff6f-061a-7390-94c3-19aa11e88eb2,kb-001,NaN
2,I'm cust_001. When does my next billing cycle ...,"{""classification"": ""billing"", ""answer"": ""Your ...","[get_customer_plan, search_kb]",None,False,False,billing,1,1,1,1.0,False,2.447602,a944bbd2-0f82-44b2-b2c1-1a2c18729dfe,019dff6f-0623-7831-a047-95bc8b718866,kb-004,"[get_customer_plan, search_kb]"
3,"Hi, I'm cust_002. I forgot my password.","{""classification"": ""account"", ""answer"": ""To re...","[get_customer_plan, search_kb]",None,False,False,account,1,1,1,1.0,True,2.736424,b6f9653d-1e73-462c-acbf-6c138cdadd70,019dff6f-0625-76f2-a0cb-2591c5d972d7,kb-002,"[get_customer_plan, search_kb]"


## Now break it — the regression demo

**A regression test catches a regression — a backward step in agent quality vs. previously-known-good behavior.**

Imagine a developer pushes a PR that "refactors the prompt for brevity" and accidentally drops the safety line. We'll simulate that:

1. Remove `Never refund a free-tier customer.` from the system prompt
2. Rebuild the agent with the broken prompt
3. Run the same `evaluate()` against the same dataset
4. Watch `refund_safety` drop on `ex-012` only — **the trap fired**

This is what the eval suite catches in CI. Without it, that prompt change ships and customers get refunded who shouldn't be.

In [41]:
# Build a BROKEN agent — same model, same tools, but the 2 safety policy lines
# stripped from the system prompt. Simulates a developer "refactoring the prompt
# for brevity" and accidentally dropping both the paid-plan rule AND the
# free-tier reminder.
#
# Why these 2 specifically (not 3): the "When you used issue_refund..." line
# is UX framing (telling customer "submitted" vs "issued") — not policy.
# Removing it doesn't add to the regression catch; empirically, removing all 3
# lets the agent rescue itself via kb-001 search_kb fallback. 2-line removal
# is the empirically-verified break.

BROKEN_PROMPT = SYSTEM_PROMPT.replace(
    "   - issue_refund if (and only if) the customer is on a paid plan AND has a legitimate refund-eligible issue per kb-001\n",
    "",
).replace(
    "Be concise. Never refund a free-tier customer.\n",
    "",
)
assert "Never refund a free-tier customer" not in BROKEN_PROMPT
assert "issue_refund if (and only if)" not in BROKEN_PROMPT
print("✓ Broken prompt: 2 lines removed (paid-plan rule + free-tier reminder)")

broken_agent = build_agent(system_prompt=BROKEN_PROMPT)

# Run experiment v2 against the SAME regression split with the broken prompt.
# Same evaluators score it. Watch refund_safety drop on ex-012 specifically.
results_v2 = evaluate(
    make_target(broken_agent),
    data=regression_data,
    evaluators=ALL_EVALUATORS,
    experiment_prefix=f"v2-removed-guardrail-{RUN_ID}",
    metadata={"run_id": RUN_ID, "variant": "removed-guardrail", "demo_set": "buildalong"},
    max_concurrency=4,
)
print(f"\n✓ Experiment v2-removed-guardrail-{RUN_ID} done.")
print("\nNow: switch to LangSmith Compare view to see v1 vs v2 side-by-side.")
print(f"  Datasets → {REGRESSION_DATASET} → Experiments tab → check v1 + v2 → Compare")
print("  → refund_safety should drop on ex-012 in v2 (the trap fired)")
print("  → other evaluators should stay at ~100% — per-dimension scoring isolates")
print("    WHICH behavior regressed, not just THAT something regressed.")
print("\nThat's the regression caught — same dataset, same split, same evaluators, one prompt change.\n")

# Inline experiment view — note `feedback.refund_safety` for ex-012 should now be 0.
results_v2.to_pandas()

✓ Broken prompt: 2 lines removed (paid-plan rule + free-tier reminder)
View the evaluation results for experiment: 'v2-removed-guardrail-20260506-2335-25037938' at:
https://smith.langchain.com/o/bcd92ec8-b810-4fa0-a0c0-a6455008401d/datasets/cffad3f2-d09b-43aa-b6df-3f8f7c7dcecf/compare?selectedSessions=98337eb9-9ca6-45d7-a6c5-69626f175ba1




0it [00:00, ?it/s]


✓ Experiment v2-removed-guardrail-20260506-2335 done.

Now: switch to LangSmith Compare view to see v1 vs v2 side-by-side.
  Datasets → support-triage-buildalong → Experiments tab → check v1 + v2 → Compare
  → refund_safety should drop on ex-012 in v2 (the trap fired)
  → other evaluators should stay at ~100% — per-dimension scoring isolates
    WHICH behavior regressed, not just THAT something regressed.

That's the regression caught — same dataset, same split, same evaluators, one prompt change.



,inputs.message,outputs.final_message,outputs.tool_calls,error,reference.should_refund,reference.should_escalate,reference.expected_classification,feedback.refund_safety,feedback.classification_correct,feedback.escalation_correctness,feedback.trajectory_superset,feedback.kb_grounding,execution_time,example_id,id,reference.expected_kb_doc_id,reference.expected_tools
0,This is RIDICULOUS. I'm cust_003 and your serv...,"{""classification"": ""other"", ""answer"": ""I've es...","[get_customer_plan, escalate]",None,False,True,technical,1,0,1,NaN,None,2.617870,65736ce5-ec14-4b3d-a6b1-fc1a33a460ba,019dff6f-4c45-7961-9ff7-2d9bdeea4687,NaN,NaN
1,I'm cust_002 and I want a refund for last month.,"{""classification"": ""billing"", ""answer"": ""Your ...","[get_customer_plan, issue_refund]",None,False,NaN,billing,0,1,1,NaN,False,2.650757,0cd4d8c0-40f2-4e83-9b0b-00c7ecbbd022,019dff6f-4c3c-7883-8aac-622ae060891c,kb-001,NaN
2,"Hi, I'm cust_002. I forgot my password.","To reset your password, please visit our passw...","[get_customer_plan, search_kb]",None,False,False,account,1,1,1,1.0,True,3.706863,b6f9653d-1e73-462c-acbf-6c138cdadd70,019dff6f-4c47-7261-8a5a-c5c5747ff284,kb-002,"[get_customer_plan, search_kb]"
3,I'm cust_001. When does my next billing cycle ...,I'd like to inform you that your next billing ...,"[get_customer_plan, search_kb]",None,False,False,billing,1,1,1,1.0,False,3.537053,a944bbd2-0f82-44b2-b2c1-1a2c18729dfe,019dff6f-4c46-7c23-82eb-10f983cec02d,kb-004,"[get_customer_plan, search_kb]"


In [42]:
# Restore the prompt — v3. Confirms the gate works in both directions:
# catch the regression (v2), then verify the fix (v3).
# In production this is the developer's "I fixed the prompt, please re-run CI" step.
restored_agent = build_agent(system_prompt=SYSTEM_PROMPT)  # original prompt — guardrail intact

results_v3 = evaluate(
    make_target(restored_agent),
    data=regression_data,
    evaluators=ALL_EVALUATORS,
    experiment_prefix=f"v3-restored-{RUN_ID}",
    metadata={"run_id": RUN_ID, "variant": "restored", "demo_set": "buildalong"},
    max_concurrency=4,
)
print(f"\n✓ Experiment v3-restored-{RUN_ID} done.")
print("  refund_safety should be back to 1.0 on ex-012 — regression resolved.")
print("\nNow in the Compare view: select v1 + v2 + v3 → all three side-by-side.")
print("  v1 green → v2 red on ex-012 only → v3 green again.")
print("  That's the full catch-and-fix arc — the eval suite is the gate that lets you")
print("  iterate on the prompt with confidence.\n")

# Inline experiment view — refund_safety for ex-012 back to 1.
results_v3.to_pandas()

View the evaluation results for experiment: 'v3-restored-20260506-2335-0f57a295' at:
https://smith.langchain.com/o/bcd92ec8-b810-4fa0-a0c0-a6455008401d/datasets/cffad3f2-d09b-43aa-b6df-3f8f7c7dcecf/compare?selectedSessions=7a4ba7e6-afd9-4f43-aa41-36715db73d56




0it [00:00, ?it/s]


✓ Experiment v3-restored-20260506-2335 done.
  refund_safety should be back to 1.0 on ex-012 — regression resolved.

Now in the Compare view: select v1 + v2 + v3 → all three side-by-side.
  v1 green → v2 red on ex-012 only → v3 green again.
  That's the full catch-and-fix arc — the eval suite is the gate that lets you
  iterate on the prompt with confidence.



,inputs.message,outputs.final_message,outputs.tool_calls,error,reference.should_refund,reference.should_escalate,reference.expected_classification,feedback.refund_safety,feedback.classification_correct,feedback.escalation_correctness,feedback.trajectory_superset,feedback.kb_grounding,execution_time,example_id,id,reference.expected_kb_doc_id,reference.expected_tools
0,This is RIDICULOUS. I'm cust_003 and your serv...,"{""classification"": ""other"", ""answer"": ""Your is...","[get_customer_plan, search_kb, escalate]",None,False,True,technical,1,0,1,NaN,None,3.585612,65736ce5-ec14-4b3d-a6b1-fc1a33a460ba,019dff6f-adea-74e3-bf30-67179156abb6,NaN,NaN
1,I'm cust_002 and I want a refund for last month.,"{""classification"": ""billing"", ""answer"": ""Unfor...","[get_customer_plan, search_kb]",None,False,NaN,billing,1,1,1,NaN,True,2.708425,0cd4d8c0-40f2-4e83-9b0b-00c7ecbbd022,019dff6f-ade7-7e11-9dac-604bb18c3383,kb-001,NaN
2,I'm cust_001. When does my next billing cycle ...,"{""classification"": ""billing"", ""answer"": ""Your ...","[get_customer_plan, search_kb]",None,False,False,billing,1,1,1,1.0,False,2.640419,a944bbd2-0f82-44b2-b2c1-1a2c18729dfe,019dff6f-adeb-7111-bdb3-866d548943a0,kb-004,"[get_customer_plan, search_kb]"
3,"Hi, I'm cust_002. I forgot my password.",You can reset your password by visiting the pa...,"[get_customer_plan, search_kb]",None,False,False,account,1,1,1,1.0,True,3.157097,b6f9653d-1e73-462c-acbf-6c138cdadd70,019dff6f-adef-7330-bfa3-2de4ceb01ea0,kb-002,"[get_customer_plan, search_kb]"


## The other half — capability evals

We've now caught a regression on the regression suite. **The capability suite is the second half of the discipline** — harder examples where the agent currently struggles. We track pass rate on these *climbing over time* as we improve the agent.

Per the [Agent Evaluation Readiness Checklist](https://www.langchain.com/blog/agent-evaluation-readiness-checklist):

> Capability evals answer "what can it do?" — start with low pass rate.
> Regression evals answer "does it still work?" — should have ~100% pass rate.
> **You need both because they serve different purposes.**

The capability dataset (`support-triage-buildalong-capability`) has 3 deliberately hard examples:

- **`cap-001`**: 3 issues in one message — refund + password + 401 errors. Agent should address all three or escalate.
- **`cap-002`**: Context-aware escalation — should the agent escalate based on tone alone, or wait for the third complaint?
- **`cap-003`**: Policy synthesis — combine information from multiple KB docs to give a coherent answer.

**Same 5 evaluators run against this dataset.** The difference is purely at the assertion layer: in CI, `test_agent_quality` hard-asserts `refund_safety == 1`; `test_agent_capability` runs the same agent + evaluators but has NO `assert`. Pass/fail doesn't gate merges; the score logs to LangSmith for trend analysis.

In [43]:
# Capability evaluation — same evaluators, harder examples, NO assertion in CI.
# 3 examples × 5 evaluators with concurrency=4 → ~30 sec wall clock.
results_capability = evaluate(
    make_target(agent),  # back to the GOOD agent (we only break it for the regression demo)
    data=CAPABILITY_DATASET,
    evaluators=ALL_EVALUATORS,
    experiment_prefix=f"capability-baseline-{RUN_ID}",
    metadata={"run_id": RUN_ID, "variant": "capability", "demo_set": "buildalong"},
    max_concurrency=4,
)
print(f"\n✓ Capability experiment done: capability-baseline-{RUN_ID}")
print(f"  Open: Datasets → {CAPABILITY_DATASET} → Experiments tab")
print("\n  Pass rate is expected to be < 100% — these are deliberately hard examples")
print("  (multi-issue handling, context-aware escalation, policy synthesis).")
print("  The score tracks improvement over time as the agent's prompt/tools improve.")
print("\n  In CI, capability tests run alongside regression in the SAME pytest invocation —")
print("  but with NO assertion. They log scores; they don't gate merges.\n")

# Inline experiment view — capability scores will likely be a mix of 0/1.
results_capability.to_pandas()

View the evaluation results for experiment: 'capability-baseline-20260506-2335-2b27a899' at:
https://smith.langchain.com/o/bcd92ec8-b810-4fa0-a0c0-a6455008401d/datasets/d45ebf0d-cd67-437e-ad01-7eca34c31a05/compare?selectedSessions=63aaca4a-3b80-4bcb-ae88-c800536640c7




0it [00:00, ?it/s]


✓ Capability experiment done: capability-baseline-20260506-2335
  Open: Datasets → support-triage-buildalong-capability → Experiments tab

  Pass rate is expected to be < 100% — these are deliberately hard examples
  (multi-issue handling, context-aware escalation, policy synthesis).
  The score tracks improvement over time as the agent's prompt/tools improve.

  In CI, capability tests run alongside regression in the SAME pytest invocation —
  but with NO assertion. They log scores; they don't gate merges.



,inputs.message,outputs.final_message,outputs.tool_calls,error,reference.should_refund,reference.expected_tools,reference.should_escalate,reference.capability_signal,reference.expected_kb_doc_ids,reference.expected_classification,feedback.refund_safety,feedback.classification_correct,feedback.escalation_correctness,feedback.trajectory_superset,feedback.kb_grounding,execution_time,example_id,id,reference.expected_kb_doc_id
0,"Hi, my account is cust_002 (currently on free ...","As you are currently on the free tier plan, yo...","[get_customer_plan, search_kb]",None,False,"[get_customer_plan, search_kb]",False,policy_synthesis,"[kb-001, kb-006]",billing,1,1,1,1,None,3.311456,3235579a-1329-4dfb-93f6-f682761fec76,019dff6f-d81b-7370-a4dc-4667c3c531b4,NaN
1,I'm cust_001. Three things — (1) I cancelled m...,"{""classification"": ""billing"", ""answer"": ""Your ...","[get_customer_plan, search_kb, search_kb, sear...",None,True,"[get_customer_plan, search_kb]",False,multi_issue_handling,"[kb-001, kb-002, kb-003]",billing,1,1,0,1,None,4.649485,16935882-3d77-456a-bfe7-c253b9ced93f,019dff6f-d840-74f1-8263-25517d9bc4c4,NaN
2,I'm cust_003 on enterprise. We're getting thro...,"{""classification"": ""technical"", ""answer"": ""I'v...","[get_customer_plan, search_kb, search_kb, esca...",None,False,"[get_customer_plan, search_kb]",True,context_aware_escalation,NaN,technical,1,1,1,1,False,3.382237,044784f6-76c4-4a3b-8818-daa8e580da9b,019dff6f-d823-76d1-a766-ae45c1f442b3,kb-003


## From notebook to CI gate

We've built dataset, agent, evaluators, experiments, and seen a regression caught — all in this notebook. **The CI gate is the same evaluators wrapped in pytest tests** — running automatically on every PR via GitHub Actions.

The wrapping is one line per test:

```python
# tests/test_agent_quality.py — abbreviated
def test_agent_quality(example):
    # ... runs agent, calls 5 evaluators, logs scores ...
    assert rs["score"] == 1, rs.get("comment")    # ← THE GATE
```

That's the entire gating mechanism. If the evaluator returns 0 → `AssertionError` → pytest exits non-zero → GitHub Actions goes red → merge button disabled.

**Capability tests look identical but have NO `assert`** — scores log to LangSmith for trend analysis, never block the PR.

The full pytest file is in `tests/test_agent_quality.py`. We won't run it live in this notebook (pytest discovers test files in directories — that's its production-style sibling); the next cell shows the same tests running in the cloud via the GitHub Actions API.

In [44]:
# Live CI status from GitHub Actions — public-repo API, no auth needed.
import requests
REPO = "tanmaiyii/langchain-eval-driven-workshop"
WORKFLOW = "evals.yml"

resp = requests.get(
    f"https://api.github.com/repos/{REPO}/actions/workflows/{WORKFLOW}/runs",
    params={"per_page": 5}, timeout=10,
)
runs = resp.json().get("workflow_runs", [])

print(f"Latest CI runs for {REPO}:\n")
for r in runs[:5]:
    icon = {"success": "✅", "failure": "❌"}.get(r.get("conclusion"), "⏳")
    print(f"  {icon} Run #{r['run_number']} — {r.get('conclusion') or r.get('status')}")
    print(f"     {r['display_title'][:65]}")
    print(f"     {r['created_at'][:19]}  |  {r['html_url']}\n")

print("Same `pytest tests/` command runs in this cloud as I would run locally.")
print("Regression assertions block the merge; capability runs alongside without gating.")

Latest CI runs for tanmaiyii/langchain-eval-driven-workshop:

  ✅ Run #10 — success
     Merge pull request #4 from tanmaiyii/add-buildalong-notebook
     2026-05-06T19:20:59  |  https://github.com/tanmaiyii/langchain-eval-driven-workshop/actions/runs/25456119333

  ✅ Run #9 — success
     Add build-along notebook + untrack notebook.ipynb
     2026-05-06T19:18:52  |  https://github.com/tanmaiyii/langchain-eval-driven-workshop/actions/runs/25456017525

  ✅ Run #8 — success
     docs+code: restore openevals; add v1/v2/v3 regression-demo script
     2026-05-05T17:48:09  |  https://github.com/tanmaiyii/langchain-eval-driven-workshop/actions/runs/25392748421

  ✅ Run #7 — success
     docs+code: align surfaces; remove unused trajectory_judge
     2026-05-05T11:52:57  |  https://github.com/tanmaiyii/langchain-eval-driven-workshop/actions/runs/25374741168

  ✅ Run #6 — success
     Merge pull request #3 from tanmaiyii/dataset-docstring-clarificat
     2026-05-05T01:02:14  |  https://github.co

## Calibration — trusting the LLM-judge

The LLM-judge isn't trustworthy until it's **calibrated against human labels**. Until then, gating CI on it would fail PRs on judge variance, not real regressions. So we track judge scores; we don't gate on them.

The calibration loop:

1. Find runs where the judge scored low or where you suspect a disagreement
2. Queue them for human review
3. Compare human scores against judge scores
4. Refine the rubric prompt against disagreements
5. Re-run; check that judge-human agreement improves

**Where prompt iteration fits.** Refining the rubric prompt is where [LangSmith Playground](https://docs.langchain.com/langsmith/playground) earns its place — Module 3 of the curriculum. Same Playground works for iterating the agent's `SYSTEM_PROMPT` against regression scores.

This cell creates the annotation queue programmatically and adds the runs flagged by our evaluators.

In [45]:
# Programmatic annotation queue creation — finds flagged runs, queues for human review.
queue_name = f"calibration-{RUN_ID}"

queue = client.create_annotation_queue(
    name=queue_name,
    description="Runs from the build-along regression demo with non-perfect scores - human review for calibration",
)
print(f"✓ Created annotation queue: {queue.name}")
print(f"  ID: {queue.id}\n")

EVAL_KEYS = ["refund_safety", "classification_correct", "kb_grounding"]
flagged = set()
for key in EVAL_KEYS:
    try:
        runs = client.list_runs(
            project_name=os.getenv("LANGSMITH_PROJECT", "agent-eval-buildalong"),
            filter=f'and(eq(feedback_key, "{key}"), lt(feedback_score, 1))',
            is_root=True, limit=10,
        )
        for r in runs: flagged.add(r.id)
        count = sum(1 for r in client.list_runs(
            project_name=os.getenv("LANGSMITH_PROJECT", "agent-eval-buildalong"),
            filter=f'and(eq(feedback_key, "{key}"), lt(feedback_score, 1))',
            is_root=True, limit=10,
        ))
        print(f"  {key:30} → flagged runs scanned")
    except Exception as e:
        print(f"  {key:30} → query failed: {type(e).__name__}")

if flagged:
    client.add_runs_to_annotation_queue(queue_id=queue.id, run_ids=list(flagged))
    print(f"\n✓ Added {len(flagged)} runs to queue {queue.name}")
    print("\nNow: switch to LangSmith → Annotation Queues → review the flagged runs.")
    print("Each row: trace on the left, evaluator scores on the right, 'Add Feedback' form for human label.")
else:
    print("No flagged runs found — all evaluators scored perfectly. Demo data needed.")

✓ Created annotation queue: calibration-20260506-2335
  ID: 295b3f83-73ab-4d7c-8a9d-e208b9c51909

  refund_safety                  → flagged runs scanned
  classification_correct         → flagged runs scanned
  kb_grounding                   → flagged runs scanned
No flagged runs found — all evaluators scored perfectly. Demo data needed.


## What we built

| | What | Where it lives |
|---|---|---|
| 1 | A dataset | LangSmith → Datasets → `support-triage-buildalong` |
| 2 | An agent + HITL middleware | This notebook (functions: `build_agent`, `make_target`) |
| 3 | Code + LLM-as-judge evaluators | This notebook (functions: `refund_safety`, `classification_correct`, `kb_grounding_judge`) |
| 4 | Two experiments — v1 baseline + v2 broken | LangSmith → Datasets → `support-triage-buildalong` → Experiments tab |
| 5 | The regression caught visually | LangSmith Compare view: refund_safety drops on ex-012 in v2 only |
| 6 | A pytest CI gate (in `tests/test_agent_quality.py`) | GitHub Actions runs on every PR; same evaluators wrapped in `assert` |
| 7 | An annotation queue for calibration | LangSmith → Annotation Queues |

## The takeaway

Eval-driven development is the discipline that turns *"we hope it works"* into *"we know it works, because the gate is locked when it doesn't."* Every prompt change becomes testable against the regression suite — instead of guess-and-check.

**The pattern transfers.** Clone the repo, swap the agent and the dataset, wire your GitHub Actions secrets. The discipline is portable; the agent we built today is the vehicle. Same loop works for sales agents, ops copilots, internal assistants — any agent with a written policy.

## Where this fits

Module 4 of a 6-module curriculum. Module 1-3 build the agent (create_agent fundamentals, tracing, prompt engineering with Playground). **Module 4 — today — gives you the offline reliability loop.** Module 5 layers online evals + the Insights Agent on top: production traces feed back into your dataset, and the loop closes.

You can't skip Module 4 to get to 5. Online evals without an offline regression suite is noise without signal.